# Train English PCM with 5-Fold Leave-One-Session-Out evaluation

This notebook trains the English PCM model to classify four IEMOCAP emotions: `hap` (Happy), `neu` (Neutral), `sad` (Sad), and `ang` (Angry). It uses five Leave-One-Session-Out folds: each complete session is held out for testing once, and the other four sessions are used for training. Fold splitting and the classification training loop are implemented in this notebook; the original PCM ZIP and its trainer are used unchanged.

## Kaggle setup
1. In **Notebook options**, select a GPU accelerator (for example, T4) and enable **Internet** so Hugging Face can download the pretrained models.
2. Add the original Kaggle input dataset containing `all_fixed.csv`, `sentences_wav/`, and `PCM/english_ver/` with the original PCM files.
3. Run the cells in order. Kaggle input folders are read-only; per-fold checkpoints, logs, metrics, and predictions are written to `/kaggle/working/pcm_classification`.

The notebook reads the dataset directly from `/kaggle/input/pcm_pcm` (no ZIP extraction needed). If Kaggle mounted it under a different path, set `DATASET_DIR_OVERRIDE` in the next cell. Training uses a fixed number of epochs, evaluates the held-out session after each epoch for reporting, and does not select checkpoints based on test scores.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled. Select a GPU accelerator in Notebook options and restart the session.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# Kaggle mounts your dataset as /kaggle/input/<dataset-slug>.
DATASET_DIR_OVERRIDE = Path('/kaggle/input/datasets/hanhmee/pcm-pcm')

input_root = Path('/kaggle/input')
if not input_root.exists():
    raise FileNotFoundError("/kaggle/input does not exist. This notebook is intended to run on Kaggle.")

DATASET_DIR = Path(DATASET_DIR_OVERRIDE) if DATASET_DIR_OVERRIDE else None
if DATASET_DIR is None or not (
    (DATASET_DIR / 'all_fixed.csv').is_file()
    and (DATASET_DIR / 'sentences_wav').is_dir()
    and (DATASET_DIR / 'PCM' / 'english_ver' / 'iemocap_audio_train.py').is_file()
):
    dataset_candidates = [
        csv_path.parent for csv_path in input_root.rglob('all_fixed.csv')
        if (csv_path.parent / 'sentences_wav').is_dir()
        and (csv_path.parent / 'PCM' / 'english_ver' / 'iemocap_audio_train.py').is_file()
    ]
    if len(dataset_candidates) != 1:
        raise FileNotFoundError(f"Expected one input with all_fixed.csv, sentences_wav/, and PCM/english_ver; found: {dataset_candidates}. Set DATASET_DIR_OVERRIDE to its Kaggle input path.")
    DATASET_DIR = dataset_candidates[0]

DATA_DIR = DATASET_DIR
SOURCE_DIR = DATASET_DIR / 'PCM' / 'english_ver'
CSV_PATH = DATA_DIR / 'all_fixed.csv'
assert (SOURCE_DIR / 'iemocap_audio_model.py').is_file(), f"Missing model file in {SOURCE_DIR}"
assert (SOURCE_DIR / 'audio_parsing_dataset.py').is_file(), f"Missing dataset loader in {SOURCE_DIR}"
assert CSV_PATH.is_file(), f"Missing CSV: {CSV_PATH}"
assert (DATA_DIR / 'sentences_wav').is_dir(), f"Missing audio directory under {DATA_DIR}"
print('Kaggle dataset:', DATASET_DIR)
print('PCM source:', SOURCE_DIR)
print('IEMOCAP data:', DATA_DIR)
print('CSV:', CSV_PATH)

## Install missing packages

This cell does not install or upgrade PyTorch. If Kaggle reports a binary/version conflict, restart the session and keep the Kaggle-provided `torch` and `torchaudio` versions.

In [ ]:
missing_packages = []
if importlib.util.find_spec('transformers') is None:
    missing_packages.append('transformers')
if importlib.util.find_spec('parselmouth') is None:
    missing_packages.append('praat-parselmouth')
if importlib.util.find_spec('sklearn') is None:
    missing_packages.append('scikit-learn')
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('Required packages are already installed.')

sys.path.insert(0, str(SOURCE_DIR))
import pandas as pd
from transformers import AutoConfig

## Validate labels, audio files, and sessions

The notebook runs 5-fold Leave-One-Session-Out (LOSO): each fold tests on one complete session (Ses01?Ses05) and trains on the other four. The held-out test session is never used to select a checkpoint.

In [ ]:
df = pd.read_csv(CSV_PATH)
required_columns = {'sample_name', 'transcript', 'emotion', 'emotion_id', 'path'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'CSV is missing required columns: {sorted(missing_columns)}')

expected_mapping = {'hap': 0, 'neu': 1, 'sad': 2, 'ang': 3}
observed_mapping = df.groupby('emotion')['emotion_id'].agg(lambda ids: set(ids.astype(int))).to_dict()
if observed_mapping != {label: {label_id} for label, label_id in expected_mapping.items()}:
    raise ValueError(f'Unexpected emotion/emotion_id mapping: {observed_mapping}')

fold_sessions = [f'Ses0{session}' for session in range(1, 6)]
sessions = df['sample_name'].astype(str).str.extract(r'^(Ses0[1-5])', expand=False)
if sessions.isna().any():
    raise ValueError('Every sample_name must start with a session ID from Ses01 to Ses05.')
session_counts = sessions.value_counts().reindex(fold_sessions, fill_value=0)
if (session_counts == 0).any():
    raise ValueError(f'Every LOSO fold must have samples in all five sessions: {session_counts.to_dict()}')

audio_paths = [DATA_DIR / Path(p) for p in df['path'].astype(str)]
missing_audio = [str(p) for p in audio_paths if not p.is_file()]
if missing_audio:
    raise FileNotFoundError(f'{len(missing_audio)} audio files are missing; first examples: {missing_audio[:5]}')

print('Rows:', len(df))
print('Label counts:')
display(df.groupby(['emotion_id', 'emotion']).size().rename('samples').to_frame())
print('Samples per held-out test session:', session_counts.to_dict())
print('Each fold trains on the other four sessions; all CSV audio paths exist.')

## Configure model and training

The Wav2Vec2 hidden size is read from its pretrained config because the pitch input must use the same feature width. Reduce batch sizes if GPU memory is insufficient. The first run downloads the Wav2Vec2 and RoBERTa tokenizer assets.

In [ ]:
from argparse import Namespace

AUDIO_MODEL = 'facebook/wav2vec2-base-960h'
model_config = AutoConfig.from_pretrained(AUDIO_MODEL)
output_dir = Path('/kaggle/working/pcm_classification')
folds_dir = output_dir / 'folds'
output_dir.mkdir(parents=True, exist_ok=True)
folds_dir.mkdir(parents=True, exist_ok=True)

args = Namespace(
    objective='classification',
    task='intonation_and_wav2vec2',
    dataset='iemocap',
    text_cap_path=str(CSV_PATH),
    audio_path=str(DATA_DIR),
    image_path=str(DATA_DIR),  # Required by the existing loader; not used by this task.
    audio_model_name=AUDIO_MODEL,
    tokenizer='roberta-base',
    text='nope',
    embedding='linear',
    hidden_size=model_config.hidden_size,
    normalized='ok',
    load_model_path=str(folds_dir / 'checkpoint'),
    logger_path=str(output_dir / 'train.log'),
    epoch=5,
    learning_rate=1e-5,
    max_grad_norm=1.0,
    train_batch_size=4,
    eval_batch_size=4,
    train_shuffle=True,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
)

print('Hidden size:', args.hidden_size)
print('Batch sizes:', args.train_batch_size, '/', args.eval_batch_size)
print('Folds:', fold_sessions)
print('Epochs per fold:', args.epoch)
print('Output directory:', output_dir)

## Train 5 Leave-One-Session-Out folds

Each fold trains on all utterances from four sessions and evaluates the held-out session after every epoch. Training runs for a fixed number of epochs; test scores are reported only and are never used to select a checkpoint. Each fold's final-epoch model and log are saved separately.

In [ ]:
import logging
import random
import sys

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Keep using the original PCM model, feature extraction and Trainer implementation.
for module_name in ('iemocap_audio_train', 'audio_parsing_dataset', 'iemocap_audio_model'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

trainer_source = (SOURCE_DIR / 'iemocap_audio_train.py').read_text(encoding='utf-8')
dataset_source = (SOURCE_DIR / 'audio_parsing_dataset.py').read_text(encoding='utf-8')
model_source = (SOURCE_DIR / 'iemocap_audio_model.py').read_text(encoding='utf-8')
required_source_markers = {
    'padded audio processing': 'padding=True' in dataset_source,
    'safe pitch normalization': 'std < 1e-6' in dataset_source and 'finite_mask' in dataset_source,
    'classification attention fix': (
        'query=pitch_embeds.transpose(0, 1)' in model_source
        and 'Cross-attention produced non-finite values for classification.' in model_source
    ),
    'classification backbone eval': 'self.audio_model.eval()' in model_source and 'non_finite_layers=' in model_source,
    'original PCM Trainer': 'class Trainer:' in trainer_source,
}
missing_markers = [name for name, present in required_source_markers.items() if not present]
if missing_markers:
    raise RuntimeError(
        f'The original PCM input under {SOURCE_DIR} is missing required code: {missing_markers}. '
        'Attach the original dataset version containing the current PCM model and dataset-loader fixes.'
    )

from audio_parsing_dataset import data_loader
from iemocap_audio_train import Trainer

fold_trainers = {}
epoch_rows = []
prediction_rows = []
label_names = {0: 'hap', 1: 'neu', 2: 'sad', 3: 'ang'}

for fold_number, test_session in enumerate(fold_sessions, start=1):
    fold_dir = folds_dir / test_session
    checkpoint_dir = fold_dir / 'checkpoint'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    train_sessions = [session for session in fold_sessions if session != test_session]
    train_rows = df[sessions != test_session].copy()
    test_rows = df[sessions == test_session].copy()
    if train_rows.empty or test_rows.empty:
        raise ValueError(f'Fold {test_session} has an empty train or test split.')

    fold_args = Namespace(**vars(args).copy())
    fold_args.load_model_path = str(checkpoint_dir)
    fold_args.logger_path = str(fold_dir / 'train.log')

    random.seed(fold_args.seed)
    np.random.seed(fold_args.seed)
    torch.manual_seed(fold_args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_args.seed)

    trainer = Trainer(fold_args)
    trainer.train_dataset = data_loader(train_rows, fold_args)
    trainer.test_dataset = data_loader(test_rows, fold_args)
    trainer.train_dataloader = DataLoader(
        trainer.train_dataset,
        batch_size=fold_args.train_batch_size,
        shuffle=fold_args.train_shuffle,
        collate_fn=trainer.train_dataset.collate_fn,
    )
    trainer.test_dataloader = DataLoader(
        trainer.test_dataset,
        batch_size=fold_args.eval_batch_size,
        shuffle=False,
        collate_fn=trainer.test_dataset.collate_fn,
    )
    trainer.set_parameters()
    trainer.epoch_metrics = []

    logger = logging.getLogger(f'pcm_loso.{test_session}')
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    log_handler = logging.FileHandler(fold_args.logger_path, encoding='utf-8')
    log_handler.setFormatter(logging.Formatter('%(asctime)s %(message)s'))
    logger.addHandler(log_handler)

    print(f'\nFold {fold_number}/5: train={train_sessions}; test={test_session}')
    try:
        for epoch in range(trainer.training_epochs):
            trainer.vad_model.train()
            total_loss = 0.0
            for batch_index, batch in enumerate(trainer.train_dataloader):
                sample_ids, _, batch_tokens, batch_attention_mask, batch_audio, _, batch_labels = batch
                batch_tokens = batch_tokens.cuda()
                batch_attention_mask = batch_attention_mask.cuda()
                pitch_audio, audio_inputs = batch_audio
                pitch_audio = pitch_audio.cuda()
                audio_inputs = audio_inputs.cuda()
                batch_labels = batch_labels.cuda().long()

                logits = trainer.vad_model(
                    pitch_audio, audio_inputs, batch_tokens, batch_attention_mask
                )
                if logits is None:
                    raise RuntimeError(f'Model returned no logits at batch {batch_index}; samples={sample_ids}.')
                if not torch.isfinite(logits).all():
                    raise FloatingPointError(
                        f'Model produced non-finite logits at batch {batch_index}; samples={sample_ids}.'
                    )
                if batch_labels.min().item() < 0 or batch_labels.max().item() >= logits.size(-1):
                    raise ValueError(
                        f'Labels outside output range at batch {batch_index}; samples={sample_ids}; '
                        f'labels={batch_labels.tolist()}; classes={logits.size(-1)}.'
                    )
                loss = F.cross_entropy(logits, batch_labels)
                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        f'Non-finite loss at batch {batch_index}; samples={sample_ids}.'
                    )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainer.vad_model.parameters(), trainer.max_grad_norm)
                trainer.optimizer.step()
                trainer.scheduler.step()
                trainer.optimizer.zero_grad(set_to_none=True)
                total_loss += loss.item()

            mean_train_loss = total_loss / len(trainer.train_dataloader)
            test_metrics, sample_ids, predictions, confidences = trainer.evaluate_classification(
                trainer.vad_model, trainer.test_dataloader
            )
            trainer.SaveModel(trainer.vad_model, trainer.save_path)
            epoch_metrics = {
                'epoch': epoch + 1,
                'train_loss': mean_train_loss,
                'test_wa': test_metrics['wa'],
                'test_ua': test_metrics['ua'],
                'test_macro_f1': test_metrics['macro_f1'],
            }
            trainer.epoch_metrics.append(epoch_metrics)
            logger.info(
                'Epoch %d/%d - train_loss=%.4f, test_wa=%.4f, test_ua=%.4f, test_macro_f1=%.4f',
                epoch + 1, trainer.training_epochs, mean_train_loss,
                test_metrics['wa'], test_metrics['ua'], test_metrics['macro_f1'],
            )
            print(
                f"Epoch {epoch + 1}/{trainer.training_epochs} - "
                f"loss={mean_train_loss:.4f}, test WA={test_metrics['wa']:.4f}, "
                f"UA={test_metrics['ua']:.4f}, Macro F1={test_metrics['macro_f1']:.4f}"
            )
            if epoch + 1 == trainer.training_epochs:
                prediction_rows = [row for row in prediction_rows if row['fold'] != test_session]
                prediction_rows.extend(
                    {
                        'fold': test_session,
                        'sample_name': sample_id,
                        'emotion': label_names[int(label_id)],
                        'confidence': float(confidence),
                    }
                    for sample_id, label_id, confidence in zip(sample_ids, predictions, confidences)
                )

        checkpoint_file = checkpoint_dir / 'model.bin'
        if not checkpoint_file.is_file():
            raise FileNotFoundError(f'Fold {test_session} finished without saving: {checkpoint_file}')
        if len(trainer.epoch_metrics) != trainer.training_epochs:
            raise RuntimeError(
                f'Fold {test_session} logged {len(trainer.epoch_metrics)} epochs; '
                f'expected {trainer.training_epochs}.'
            )
        fold_trainers[test_session] = trainer
        for metrics in trainer.epoch_metrics:
            epoch_rows.append({'fold': test_session, **metrics})
    finally:
        logger.removeHandler(log_handler)
        log_handler.close()

fold_metrics = pd.DataFrame(epoch_rows)
fold_metrics_path = output_dir / 'loso_fold_metrics.csv'
fold_metrics.to_csv(fold_metrics_path, index=False)
metric_columns = ['train_loss', 'test_wa', 'test_ua', 'test_macro_f1']
fold_summary = fold_metrics.groupby('epoch')[metric_columns].agg(['mean', 'std'])
print('Per-epoch mean and standard deviation across the five held-out sessions:')
display(fold_summary)
print('Per-fold, per-epoch metrics saved:', fold_metrics_path)

predictions_path = output_dir / 'loso_test_predictions.csv'
predictions_df = pd.DataFrame(prediction_rows)
if len(predictions_df) != len(df) or predictions_df['sample_name'].nunique() != len(df):
    raise RuntimeError(
        f'Expected one held-out prediction for each of {len(df)} utterances; '
        f'got {len(predictions_df)} rows and {predictions_df["sample_name"].nunique()} unique utterances.'
    )
predictions_df.to_csv(predictions_path, index=False)
print('LOSO test predictions saved:', predictions_path)

## Download metrics and held-out predictions

Each fold's final-epoch predictions are collected during training. The notebook checks that every utterance appears exactly once in the held-out predictions.

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(fold_metrics_path)))
display(FileLink(str(predictions_path)))

## Download artifacts

Download `loso_fold_metrics.csv` for metrics from every epoch and fold, and `loso_test_predictions.csv` for held-out predictions. Each fold's checkpoint and training log are in `/kaggle/working/pcm_classification/folds/SesXX/`. The original PCM ZIP does not need to be modified.